In [1]:
import gc
from datetime import datetime

import polars as pl
import lightgbm as lgb
import numpy as np


from src.util.common import mean_grouped_spearman_correlation, get_constant_features, get_redundant_features
from src.util.constants import DATA_PATH, FIXED_LGB_PARAMETERS
from util.common import load_from_pickle

In [2]:
df_meta_model = pl.read_parquet(f"{DATA_PATH}/folds/df_meta_model.parquet")
number_of_meta_model_eras = len(df_meta_model["era"].unique())
use_meta_model = True
if number_of_meta_model_eras < 52/2:
    print("Less than half a year of eras - not enough to analyse!")
    use_meta_model = False

Less than half a year of eras - not enough to analyse!


In [3]:
feature_names = load_from_pickle(DATA_PATH / "raw/feature_names.pkl")

### Linear model

In [3]:
# We should be able to remove constant features for each fold.
# However, if we remove features by fold, we run into matrix singularity issues.
# As a quickfix, we remove the features that are problematic on the first fold across all folds.
df_train = pl.read_parquet(f"{DATA_PATH}/folds/df_train_0.parquet")

constant_features = get_constant_features(df_train)
redundant_features = get_redundant_features(df_train)
problematic_features = set(constant_features).union(set(redundant_features))
print(f"Found {len(problematic_features)} problematic features to exclude.")
del df_train

kept_feature_names = [col for col in feature_names if col not in problematic_features]
feature_mask = np.array([col not in problematic_features for col in feature_names])

Features: 100%|██████████| 3555/3555 [30:32<00:00,  1.94it/s]


Found 3 candidate pairs for duplicated features...


Candidate pairs: 100%|██████████| 3/3 [00:00<00:00, 126.25it/s]


In [6]:
df_result_linear = pl.DataFrame()

for fold in range(3):
    X = load_from_pickle(DATA_PATH / f"folds/X_sampled_train_{fold}.pkl")[:, feature_mask]
    y = load_from_pickle(DATA_PATH / f"folds/y_sampled_train_{fold}.pkl")
    X = np.hstack([np.ones((X.shape[0], 1)), X])

    beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
    del X, y

    df_validate = pl.read_parquet(f"{DATA_PATH}/folds/df_validate_{fold}.parquet")
    X_validate = df_validate[kept_feature_names].to_numpy()
    X_validate = np.hstack([np.ones((X_validate.shape[0], 1)), X_validate])
    df_validate = df_validate.with_columns(
        prediction=X_validate @ beta_hat
    )

    if use_meta_model:
        X_meta_model = df_meta_model[kept_feature_names].to_numpy()
        X_meta_model = np.hstack([np.ones((X_meta_model.shape[0], 1)), X_meta_model])
        df_meta_model_with_prediction = df_meta_model.with_columns(
            prediction=X_meta_model @ beta_hat
        )

    corr = mean_grouped_spearman_correlation(
        df_validate["prediction"],
        df_validate["target"],
        df_validate["era"]
    )
    corr_w_mm = None
    if use_meta_model:
        corr_w_mm = df_meta_model_with_prediction.select(
            pl.corr("prediction", "numerai_meta_model", method="spearman")
            .over("era", mapping_strategy="explode")
        ).mean()[0, 0]

    df_result_linear = df_result_linear.vstack(pl.DataFrame({
        "fold": fold,
        "corr": corr,
        "corr_w_mm": corr_w_mm,
    }))

    del df_validate
    if use_meta_model:
        del df_meta_model_with_prediction

    print(f"Fold {fold} done.")

Fold 0 done.
Fold 1 done.
Fold 2 done.


In [12]:
df_result_linear

fold,corr,corr_w_mm
i64,f64,null
0,0.030818,null
1,0.03383,null
2,0.02324,null


In [11]:
# df_result_linear =  pl.DataFrame({
#     "fold": [0, 1, 2],
#     "corr": [0.030818, 0.03383, 0.02324],
#     "corr_w_mm": [None, None, None],
# })

### Numerai models

In [4]:
# Parameters from Numerai (compare https://docs.numer.ai/numerai-tournament/models)
parameter_dict = {
    "standard_large_lgbm": {
        "learning_rate": 0.001,
        "max_depth": 6,
        "num_leaves": 2 ** 6,
        "colsample_bytree": 0.1,
    },
    "deep_lgbm": {
        "learning_rate": 0.001,
        "max_depth": 10,
        "num_leaves": 1024,
        "colsample_bytree": 0.1,
        "min_data_in_leaf": 10000
    }
}
num_boost_round_dict = {
    "standard_large_lgbm": 20000,
    "deep_lgbm": 30000,
}

In [5]:
df_result_numerai_lgb = pl.DataFrame()
model_deep_2 = None

# for model_type in parameter_dict:
for model_type in ["deep_lgbm"]:  # FIXME
    print(f"Running model {model_type}...")
    parameters = {
        **FIXED_LGB_PARAMETERS,
        **parameter_dict[model_type]
    }

    # for fold in range(3):
    for fold in [0, 2]:  # FIXME
        lgb_train = lgb.Dataset(
            data=load_from_pickle(DATA_PATH / f"folds/X_sampled_train_{fold}.pkl"),
            label=load_from_pickle(DATA_PATH / f"folds/y_sampled_train_{fold}.pkl")
        )

        print(f"Fitting model {model_type} for fold {fold}...")
        model = lgb.train(
            params=parameters,
            train_set=lgb_train,
            num_boost_round=num_boost_round_dict[model_type]
        )
        del lgb_train
        gc.collect()

        df_validate = pl.read_parquet(f"{DATA_PATH}/folds/df_validate_{fold}.parquet")

        df_validate = df_validate.with_columns(
            prediction=model.predict(df_validate[feature_names].to_numpy())
        )
        if use_meta_model:
            df_meta_model_with_prediction = df_meta_model.with_columns(
                prediction=model.predict(df_meta_model[feature_names].to_numpy())
            )

        corr = mean_grouped_spearman_correlation(
            df_validate["prediction"],
            df_validate["target"],
            df_validate["era"]
        )
        del df_validate

        corr_w_mm = None
        if use_meta_model:
            corr_w_mm = df_meta_model_with_prediction.select(
                pl.corr("prediction", "numerai_meta_model", method="spearman")
                .over("era", mapping_strategy="explode")
            ).mean()[0, 0]

        df_result_numerai_lgb = df_result_numerai_lgb.vstack(pl.DataFrame({
            "type": model_type,
            "fold": fold,
            "corr": corr,
            "corr_w_mm": corr_w_mm
        }))

        if model_type == "deep_lgbm" and fold == 2 and use_meta_model:
            model_deep_2 = model

        if use_meta_model:
            del df_meta_model_with_prediction

        print(f"{datetime.now().strftime('%H:%M:%S')} . . . Type {model_type}, fold {fold} done. Correlation: {corr}")

Running model deep_lgbm...
Fitting model deep_lgbm for fold 0...
02:06:34 . . . Type deep_lgbm, fold 0 done. Correlation: 0.057220918143319935
Fitting model deep_lgbm for fold 2...
06:38:59 . . . Type deep_lgbm, fold 2 done. Correlation: 0.032989762930709744


In [9]:
df_result_numerai_lgb

type,fold,corr,corr_w_mm
str,i64,f64,null
"""standard_large_lgbm""",0,0.054793,null
"""standard_large_lgbm""",1,0.04736,null
"""standard_large_lgbm""",2,0.029801,null
"""deep_lgbm""",0,0.057221,null
"""deep_lgbm""",1,0.052511,null
"""deep_lgbm""",2,0.03299,null


In [8]:
# df_result_numerai_lgb =  pl.DataFrame({
#     "type": ["standard_large_lgbm", "standard_large_lgbm", "standard_large_lgbm", "deep_lgbm", "deep_lgbm", "deep_lgbm"],
#     "fold": [0, 1, 2, 0, 1, 2],
#     "corr": [0.054792887733211185, 0.047359639385524614, 0.029801456126212003, 0.057220918143319935, 0.05251083141854189, 0.032989762930709744],
#     "corr_w_mm": [None, None, None, None, None, None],
# })

In [ ]:
# MMC approximation
if use_meta_model:
    performance_meta_model = mean_grouped_spearman_correlation(
        df_meta_model["numerai_meta_model"],
        df_meta_model["target"],
        df_meta_model["era"]
    )
    # TODO: filter features for redundancy here as well
    feature_names = [x for x in df_meta_model.columns if "feature" in x]
    df_meta_model_with_prediction = df_meta_model.with_columns(
        prediction=model_deep_2.predict(df_meta_model[feature_names].to_numpy())
    )
    performance_deep_model = mean_grouped_spearman_correlation(
        df_meta_model_with_prediction["prediction"],
        df_meta_model_with_prediction["target"],
        df_meta_model_with_prediction["era"]
    )
    ratio = performance_meta_model / performance_deep_model

    df_result_numerai_lgb.filter(pl.col("type") == "deep_lgbm").select("corr") * ratio
else:
    print("Nothing to analyse, not enough meta-model eras!")

In [ ]:
if use_meta_model:
    meta_model_performance = [.044, .037, .025]  # TODO: automate

    # calculate performance with updated meta-model performance approximation
    df_result_numerai_lgb = df_result_numerai_lgb.with_columns(
        mmc_approximation = (pl.col("corr") - (pl.col("corr_w_mm") * pl.col("fold").map_elements(lambda x: meta_model_performance[x], return_dtype=pl.Float64)))
    )
    df_result_numerai_lgb = df_result_numerai_lgb.with_columns(
        performance = .75 * pl.col("corr") + 2.25 * pl.col("mmc_approximation")
    )
    print(df_result_numerai_lgb)
else:
    print("Nothing to analyse, not enough meta-model eras!")

In [ ]:
if use_meta_model:
    # let"s also check the linear model
    # calculate performance with updated meta-model performance approximation
    df_result_linear = df_result_linear.with_columns(
        mmc_approximation = (pl.col("corr") - (pl.col("corr_w_mm") * pl.col("fold").map_elements(lambda x: meta_model_performance[x], return_dtype=pl.Float64)))
    )
    df_result_linear = df_result_linear.with_columns(
        performance = .5 * pl.col("corr") + 2 * pl.col("mmc_approximation")
    )
    print(df_result_linear)
else:
    print("Nothing to analyse, not enough meta-model eras!")

In [13]:
(DATA_PATH / "results").mkdir(parents=True, exist_ok=True)
df_result_linear.write_parquet(f"{DATA_PATH}/results/df_result_linear.parquet")
df_result_numerai_lgb.write_parquet(f"{DATA_PATH}/results/df_result_numerai_lgb.parquet")

In [ ]:
# TODO: document deep benchmark corr